# Uber NYC Hot-Zones — Temporal Analysis & Year-over-Year Comparison

## Objective
Analyze when and where Uber demand peaks, and how patterns evolved from 2014 to 2015.

## Data sources
- **2014:** ~4.5M pickups clustered into k hot-zones (from NB03)
- **2015:** ~14M pickups mapped to 263 NYC taxi zones
- **Common reference:** NYC TLC taxi zone system (bridges GPS-based 2014 clusters to zone-based 2015 data)

## Sections
1. Hot-zone temporal patterns (2014 clusters)
2. Top hot-zones by time period
3. Year-over-year comparison at zone level (April–June overlap)
4. Driver recommendations

In [16]:
import sys
import warnings

sys.path.insert(0, "src")
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import DAY_NAMES, OUTPUT_DIR

In [17]:
import logging

logger = logging.getLogger(__name__)

# 2014 clustered data
df_2014 = pd.read_parquet(OUTPUT_DIR / "clustered_2014.parquet")
centers = pd.read_csv(OUTPUT_DIR / "kmeans_hotzone_centers.csv")
profiles = pd.read_csv(OUTPUT_DIR / "cluster_profiles.csv")

# Drop rows with no zone assignment (GPS points outside all TLC polygons)
n_before = len(df_2014)
df_2014 = df_2014.dropna(subset=["LocationID"])
n_dropped = n_before - len(df_2014)
logger.info(
    f"Dropped {n_dropped:,} rows with NaN LocationID "
    f"({n_dropped / n_before * 100:.2f}% of 2014 data)"
)

print(f"2014: {len(df_2014):,} rows, {df_2014['cluster'].nunique()} clusters")

# 2015 preprocessed data — MEMORY: ~14M rows, do NOT copy
df_2015 = pd.read_parquet(OUTPUT_DIR / "preprocessed_2015.parquet")
print(f"2015: {len(df_2015):,} rows")

# Zone centroids (for 2015 map visualization)
zone_centroids = pd.read_csv(OUTPUT_DIR / "zone_centroids.csv")
print(f"Zone centroids: {len(zone_centroids)} zones")

2014: 4,443,244 rows, 11 clusters
2015: 14,264,215 rows
Zone centroids: 260 zones


## 1. Hot-Zone Temporal Patterns (2014)

### 1.1 Activity by Hour of Day

Which hot-zones are most active at each hour? This heatmap shows cluster activity 
across the 24-hour cycle.

In [18]:
# Cluster activity by hour
cluster_hour = df_2014.groupby(["cluster", "hour"]).size().reset_index(name="count")
pivot = cluster_hour.pivot(index="cluster", columns="hour", values="count").fillna(0)

# Sort clusters by total activity for readability
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=False).drop(columns="total")

fig = px.imshow(
    pivot,
    labels=dict(x="Hour of Day", y="Cluster", color="Pickups"),
    x=[str(h) for h in range(24)],
    color_continuous_scale="YlOrRd",
    title="Hot-Zone Activity by Hour — 2014",
    height=max(400, len(pivot) * 15),
    aspect="auto",
)
fig.write_image("reports/figures/04_01_cluster_hourly_heatmap.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![04_01_cluster_hourly_heatmap](reports/figures/04_01_cluster_hourly_heatmap.png)

**Observation:**

Midtown Center, Union Sq, and TriBeCa concentrate the vast majority of pickups, reaching 80k-100k+ during peak hours, while clusters 4-8 remain an order of magnitude lower. Activity ramps up from 7 AM, peaks sharply between 14:00 and 19:00 (darkest cells at hours 16-17 in Midtown Center), and drops to a universal trough from 2:00 to 5:00 AM. This three-tier structure (high / medium / low clusters) persists across all hours, indicating that hot-zone rank is driven primarily by location rather than time of day.

### 1.2 Activity by Day of Week

In [19]:
cluster_day = df_2014.groupby(["cluster", "day_of_week"]).size().reset_index(name="count")
pivot_day = cluster_day.pivot(index="cluster", columns="day_of_week", values="count").fillna(0)

# Sort by total
pivot_day["total"] = pivot_day.sum(axis=1)
pivot_day = pivot_day.sort_values("total", ascending=False).drop(columns="total")

fig = px.imshow(
    pivot_day,
    labels=dict(x="Day of Week", y="Cluster", color="Pickups"),
    x=DAY_NAMES,
    color_continuous_scale="YlOrRd",
    title="Hot-Zone Activity by Day of Week — 2014",
    height=max(400, len(pivot_day) * 15),
    aspect="auto",
)
fig.write_image("reports/figures/04_02_cluster_daily_heatmap.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![04_02_cluster_daily_heatmap](reports/figures/04_02_cluster_daily_heatmap.png)

**Observation:**

The same three clusters (Midtown Center, Union Sq, TriBeCa) dominate every day of the week, with Wednesday and Thursday showing the highest volumes (darkest cells exceeding 200k pickups in Midtown Center). Weekend activity drops noticeably — Sunday is the lightest day across all clusters. The weekday-vs-weekend contrast is most pronounced in the top clusters, suggesting that the busiest hot-zones are driven by commuter and business demand rather than leisure.

## 2. Top Hot-Zones by Time Period

We identify the busiest clusters during key time windows that matter for driver positioning.

In [20]:
time_periods = {
    "Morning Rush (7–9 AM)": (7, 9),
    "Midday (11 AM–2 PM)": (11, 14),
    "Evening Rush (5–7 PM)": (17, 19),
    "Late Night (10 PM–2 AM)": (22, 2),
}

for period_name, (start, end) in time_periods.items():
    if start < end:
        mask = df_2014["hour"].between(start, end - 1)
    else:  # wraps midnight
        mask = (df_2014["hour"] >= start) | (df_2014["hour"] < end)
    
    period_data = df_2014[mask]
    top = (
        period_data.groupby("cluster")
        .size()
        .reset_index(name="pickups")
        .sort_values("pickups", ascending=False)
        .head(5)
        .merge(profiles[["cluster", "dominant_zone", "dominant_borough", "lat", "lon"]], on="cluster")
    )
    
    print(f"\n{'='*60}")
    print(f"  {period_name}")
    print(f"{'='*60}")
    print(top.to_string(index=False))


  Morning Rush (7–9 AM)
 cluster  pickups         dominant_zone dominant_borough       lat        lon
       5    86813              Union Sq        Manhattan 40.740630 -73.994800
       8    76982 Upper East Side North        Manhattan 40.781826 -73.960046
       0    74184        Midtown Center        Manhattan 40.759264 -73.978665
       1    62884  TriBeCa/Civic Center        Manhattan 40.719379 -74.000937
      10    26516            Park Slope         Brooklyn 40.680383 -73.976126

  Midday (11 AM–2 PM)
 cluster  pickups         dominant_zone dominant_borough       lat        lon
       0   135064        Midtown Center        Manhattan 40.759264 -73.978665
       5   107389              Union Sq        Manhattan 40.740630 -73.994800
       1    97189  TriBeCa/Civic Center        Manhattan 40.719379 -74.000937
       8    63204 Upper East Side North        Manhattan 40.781826 -73.960046
      10    34887            Park Slope         Brooklyn 40.680383 -73.976126

  Evening Rush 

**Interpretation:**
- **Morning rush** concentrates near business districts and transit hubs — commuters heading to work
- **Evening rush** peaks in office areas (Midtown, Financial District) — workers heading home or out
- **Late night** shifts toward entertainment and nightlife districts (Union Sq, TriBeCa, Williamsburg) and residential areas like Park Slope

These patterns directly inform driver positioning: a driver should be near nightlife districts late at night 
and near Midtown/FiDi during rush hours.

## 3. Year-over-Year Comparison

### 3.1 Approach

The 2014 and 2015 datasets have different formats:
- **2014:** GPS coordinates → clustered → mapped to taxi zones via spatial join
- **2015:** Zone IDs directly from the TLC system

The common reference frame is the **taxi zone** (263 zones). We compare at this level.

For a fair comparison, we focus on the **overlapping months: April, May, June** — 
the only period covered by both datasets.

In [21]:
# Filter to April-June for fair comparison
# 2014: plain boolean filter — no .copy() needed (only aggregations follow, no item assignment)
overlap_2014 = df_2014[df_2014["month"].isin([4, 5, 6])]
# 2015: view is safe — MEMORY: never copy the 14M-row frame
overlap_2015 = df_2015[df_2015["month"].isin([4, 5, 6])]

print(f"Overlapping period (Apr-Jun):")
print(f"  2014: {len(overlap_2014):,} pickups")
print(f"  2015: {len(overlap_2015):,} pickups")
print(f"  Growth: {(len(overlap_2015) / len(overlap_2014) - 1) * 100:.1f}%")

Overlapping period (Apr-Jun):
  2014: 1,847,276 pickups
  2015: 7,789,614 pickups
  Growth: 321.7%


### 3.2 Overall Volume Growth

Uber grew significantly between 2014 and 2015. We quantify this growth and check 
whether it was uniform across boroughs.

In [22]:
monthly_14 = overlap_2014.groupby("month").size().reset_index(name="count")
monthly_14["year"] = "2014"
monthly_15 = overlap_2015.groupby("month").size().reset_index(name="count")
monthly_15["year"] = "2015"
monthly = pd.concat([monthly_14, monthly_15])

fig = px.bar(
    monthly, x="month", y="count", color="year", barmode="group",
    title="Monthly Pickups — Apr-Jun 2014 vs 2015",
    labels={"month": "Month", "count": "Pickups", "year": "Year"},
    color_discrete_map={"2014": "#636EFA", "2015": "#EF553B"},
    height=400,
)
fig.update_xaxes(dtick=1)
fig.write_image("reports/figures/04_03_monthly_volume_comparison.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![04_03_monthly_volume_comparison](reports/figures/04_03_monthly_volume_comparison.png)

**Observation:**

Uber pickups grew roughly 4x year-over-year across all three overlapping months, from ~0.5-0.65M in 2014 to ~2.3-2.8M in 2015 (321.7% overall growth). The 2014 volumes remain nearly flat month to month, while 2015 shows an upward trend from April (~2.3M) to June (~2.8M), suggesting accelerating adoption within the year. This explosive growth sets the context for all subsequent comparisons: any structural shift in patterns occurs on top of a massive volume increase.

### 3.3 Borough-Level Evolution

In [23]:
# Standardize borough column names
# 2014: lowercase "borough" (from shapefile spatial join)
boro_14 = overlap_2014.groupby("borough").size().reset_index(name="count")
boro_14["year"] = "2014"
boro_14["pct"] = boro_14["count"] / boro_14["count"].sum() * 100

# 2015: lowercase "borough" (from map_2015_zones)
boro_15 = overlap_2015.groupby("borough").size().reset_index(name="count")
boro_15["year"] = "2015"
boro_15["pct"] = boro_15["count"] / boro_15["count"].sum() * 100

boro = pd.concat([boro_14, boro_15])

fig = px.bar(
    boro, x="borough", y="pct", color="year", barmode="group",
    title="Pickup Share by Borough — Apr-Jun 2014 vs 2015 (%)",
    labels={"borough": "Borough", "pct": "% of Total", "year": "Year"},
    color_discrete_map={"2014": "#636EFA", "2015": "#EF553B"},
    height=400,
)
fig.write_image("reports/figures/04_04_borough_share_comparison.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![04_04_borough_share_comparison](reports/figures/04_04_borough_share_comparison.png)

**Observation:**

Manhattan dominates both years but its share drops substantially from ~80% in 2014 to ~71% in 2015, a roughly 9 percentage-point decline. Brooklyn and Queens absorb most of the redistribution, rising from ~11% to ~17% and from ~7% to ~10% respectively. The Bronx, EWR, and Staten Island remain below 2% in both years. This geographic diversification indicates that Uber's growth was disproportionately driven by outer-borough adoption rather than further Manhattan saturation.

### 3.4 Top Zones — Activity Comparison

Which taxi zones saw the most pickups, and how did their ranking change between years?

In [24]:
# Zone-level aggregation (Apr-Jun only)
# 2014: use LocationID from spatial join
zones_14 = overlap_2014.groupby("LocationID").size().reset_index(name="count_2014")

# 2015: use locationID directly (note: lowercase 'l' in original column)
col_2015 = "locationID" if "locationID" in overlap_2015.columns else "LocationID"
zones_15 = overlap_2015.groupby(col_2015).size().reset_index(name="count_2015")
zones_15 = zones_15.rename(columns={col_2015: "LocationID"})

# Merge
zone_comparison = zones_14.merge(zones_15, on="LocationID", how="outer").fillna(0)
zone_comparison = zone_comparison.merge(
    zone_centroids[["LocationID", "zone", "borough"]],
    on="LocationID", how="left",
)

# Compute growth
zone_comparison["growth_pct"] = np.where(
    zone_comparison["count_2014"] > 0,
    (zone_comparison["count_2015"] / zone_comparison["count_2014"] - 1) * 100,
    np.nan,
)

# Rank change
zone_comparison["rank_2014"] = zone_comparison["count_2014"].rank(ascending=False, method="min")
zone_comparison["rank_2015"] = zone_comparison["count_2015"].rank(ascending=False, method="min")
zone_comparison["rank_change"] = zone_comparison["rank_2014"] - zone_comparison["rank_2015"]

# Show top 20 by 2015 volume
top_20 = zone_comparison.nlargest(20, "count_2015")

fig = go.Figure()
fig.add_trace(go.Bar(name="2014", x=top_20["zone"], y=top_20["count_2014"], marker_color="#636EFA"))
fig.add_trace(go.Bar(name="2015", x=top_20["zone"], y=top_20["count_2015"], marker_color="#EF553B"))
fig.update_layout(
    barmode="group",
    title="Top 20 Zones by Pickup Volume — Apr-Jun",
    xaxis_title="Taxi Zone",
    yaxis_title="Pickups",
    height=500,
    xaxis_tickangle=-45,
)
fig.write_image("reports/figures/04_05_top20_zones.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![04_05_top20_zones](reports/figures/04_05_top20_zones.png)

**Observation:**

Midtown Center leads in both years with ~85k (2014) and ~235k (2015) pickups, followed by East Village, Union Square, and TriBeCa/Civic Center. All top 20 zones show massive absolute growth while largely preserving their relative ranking — 8 of the top 10 zones are shared across years. Notably, JFK Airport and LaGuardia Airport appear in the top 20 for 2015, confirming sustained airport demand. The growth multiplier is fairly consistent across zones (roughly 2.5-4x), reinforcing that the volume surge was broad-based rather than concentrated in a few zones.

In [25]:
# Zones with highest growth (minimum 1000 pickups in 2014 for significance)
significant = zone_comparison[zone_comparison["count_2014"] >= 1000]

print("Top 10 fastest-growing zones (Apr-Jun, min 1000 pickups in 2014):")
print(significant.nlargest(10, "growth_pct")[["zone", "borough", "count_2014", "count_2015", "growth_pct"]].to_string(index=False))

print("\nTop 10 zones with biggest rank improvement:")
print(significant.nlargest(10, "rank_change")[["zone", "borough", "rank_2014", "rank_2015", "rank_change"]].to_string(index=False))

Top 10 fastest-growing zones (Apr-Jun, min 1000 pickups in 2014):
                    zone   borough  count_2014  count_2015  growth_pct
         Jackson Heights    Queens      1426.0       21854 1432.538569
               Ridgewood    Queens      1109.0       15015 1253.922453
Washington Heights North Manhattan      1611.0       21306 1222.532588
            Forest Hills    Queens      1917.0       24386 1172.091810
             Old Astoria    Queens      1352.0       15352 1035.502959
Washington Heights South Manhattan      2548.0       28560 1020.879121
    Flatbush/Ditmas Park  Brooklyn      2495.0       27331  995.430862
        Hamilton Heights Manhattan      1825.0       19920  991.506849
               Bay Ridge  Brooklyn      2012.0       21214  954.373757
                 Astoria    Queens      4894.0       50880  939.640376

Top 10 zones with biggest rank improvement:
                zone   borough  rank_2014  rank_2015  rank_change
             Astoria    Queens       72.0 

### 3.5 Temporal Pattern Evolution

Did the hourly and daily patterns change between years?

In [26]:
hourly_14 = overlap_2014.groupby("hour").size().reset_index(name="count")
hourly_14["pct"] = hourly_14["count"] / hourly_14["count"].sum() * 100
hourly_14["year"] = "2014"

hourly_15 = overlap_2015.groupby("hour").size().reset_index(name="count")
hourly_15["pct"] = hourly_15["count"] / hourly_15["count"].sum() * 100
hourly_15["year"] = "2015"

hourly = pd.concat([hourly_14, hourly_15])

fig = px.line(
    hourly, x="hour", y="pct", color="year",
    title="Hourly Distribution — Apr-Jun 2014 vs 2015 (normalized %)",
    labels={"hour": "Hour", "pct": "% of Daily Pickups", "year": "Year"},
    color_discrete_map={"2014": "#636EFA", "2015": "#EF553B"},
    height=400,
)
fig.update_xaxes(dtick=1)
fig.write_image("reports/figures/04_06_hourly_distribution_overlay.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![04_06_hourly_distribution_overlay](reports/figures/04_06_hourly_distribution_overlay.png)

**Observation:**

Both years share the same general shape — a trough from 2-5 AM and a rise through the afternoon — but the profiles diverge significantly in the evening. In 2014, the distribution spikes sharply at hour 17 (~7.8%) then drops steeply by 23:00 (~3.8%), whereas 2015 shows a flatter plateau from 17:00 to 22:00 (~6-6.7%). The 2015 curve also carries higher relative share during late-night hours (midnight ~4.5% vs ~2% in 2014). This shift from a peaked to a plateau pattern suggests Uber expanded into nightlife and late-evening use cases by 2015, consistent with the peak hour moving from 17:00 to 19:00.

In [27]:
daily_14 = overlap_2014.groupby("day_of_week").size().reset_index(name="count")
daily_14["pct"] = daily_14["count"] / daily_14["count"].sum() * 100
daily_14["year"] = "2014"
daily_14["day_name"] = daily_14["day_of_week"].map(lambda x: DAY_NAMES[x])

daily_15 = overlap_2015.groupby("day_of_week").size().reset_index(name="count")
daily_15["pct"] = daily_15["count"] / daily_15["count"].sum() * 100
daily_15["year"] = "2015"
daily_15["day_name"] = daily_15["day_of_week"].map(lambda x: DAY_NAMES[x])

daily = pd.concat([daily_14, daily_15])

fig = px.bar(
    daily, x="day_name", y="pct", color="year", barmode="group",
    title="Daily Distribution — Apr-Jun 2014 vs 2015 (normalized %)",
    labels={"day_name": "Day", "pct": "% of Weekly Pickups", "year": "Year"},
    color_discrete_map={"2014": "#636EFA", "2015": "#EF553B"},
    category_orders={"day_name": DAY_NAMES},
    height=400,
)
fig.write_image("reports/figures/04_07_daily_distribution.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![04_07_daily_distribution](reports/figures/04_07_daily_distribution.png)

**Observation:**

In 2014, the weekly distribution is skewed toward weekdays — Thursday and Friday each capture ~17.5% of weekly pickups while Sunday falls to ~10%. By 2015, the distribution flattens: Saturday becomes the peak day (~16.7%), Sunday rises to ~14%, and weekday shares compress. The weekday share drops from 76.3% to 69.5%, confirming that Uber's growth disproportionately captured weekend demand. This is consistent with the broader late-night and outer-borough expansion observed in other charts.

### 3.6 Geographic Evolution

How did the spatial distribution of pickups change? We compare zone-level density 
using normalized proportions (to account for overall volume growth).

In [28]:
# Compute zone proportions for both years
zone_props = zone_comparison.assign(
    prop_2014=zone_comparison["count_2014"] / zone_comparison["count_2014"].sum() * 100,
    prop_2015=zone_comparison["count_2015"] / zone_comparison["count_2015"].sum() * 100,
)
zone_props = zone_props.assign(prop_change=zone_props["prop_2015"] - zone_props["prop_2014"])

# Merge with centroids for mapping
zone_map = zone_props.merge(zone_centroids[["LocationID", "centroid_lat", "centroid_lon"]], on="LocationID", how="left")
zone_map = zone_map.dropna(subset=["centroid_lat", "centroid_lon"])

# Side-by-side maps
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("2014 Zone Proportions", "2015 Zone Proportions"),
    specs=[[{"type": "map"}, {"type": "map"}]],
)

for col_idx, (year_col, color_label) in enumerate(
    [("prop_2014", "2014 %"), ("prop_2015", "2015 %")], start=1
):
    fig.add_trace(
        go.Scattermap(
            lat=zone_map["centroid_lat"],
            lon=zone_map["centroid_lon"],
            mode="markers",
            marker=dict(
                size=zone_map[year_col].clip(0, 5) * 8,
                color=zone_map[year_col],
                colorscale="YlOrRd",
                showscale=(col_idx == 2),
                colorbar=dict(title=color_label) if col_idx == 2 else None,
            ),
            text=zone_map["zone"],
            hovertemplate="%{text}<br>" + color_label + ": %{marker.color:.2f}%<extra></extra>",
            name=f"{year_col[:4]}",
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    height=500,
    title_text="Zone Pickup Proportions — Apr-Jun 2014 vs 2015",
    map=dict(style="open-street-map", center=dict(lat=40.73, lon=-73.98), zoom=10),
    map2=dict(style="open-street-map", center=dict(lat=40.73, lon=-73.98), zoom=10),
)
fig.write_image("reports/figures/04_08_zone_proportion_maps.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![04_08_zone_proportion_maps](reports/figures/04_08_zone_proportion_maps.png)

**Observation:**

Both maps show heavy concentration in Manhattan (large, dark circles reaching 3-4% of total pickups per zone). The 2015 map displays visibly more and larger dots in Brooklyn, Queens, and upper Manhattan compared to 2014, confirming the geographic diversification identified in the borough share analysis. While the core Manhattan hot-zones remain dominant in both years, the spatial footprint of Uber demand clearly expanded outward by 2015, with outer-borough zones growing from near-invisible to meaningful proportions.

## 4. Key Findings & Driver Recommendations

### Findings

In [29]:
print("=" * 60)
print("  KEY FINDINGS")
print("=" * 60)

# Overall growth
print(f"\n1. VOLUME GROWTH (Apr-Jun):")
print(f"   2014: {len(overlap_2014):,} pickups")
print(f"   2015: {len(overlap_2015):,} pickups")
print(f"   Growth: {(len(overlap_2015) / len(overlap_2014) - 1) * 100:.1f}%")

# Top zone stability
top10_14 = set(zone_comparison.nlargest(10, "count_2014")["LocationID"])
top10_15 = set(zone_comparison.nlargest(10, "count_2015")["LocationID"])
overlap_top = len(top10_14 & top10_15)
print(f"\n2. TOP ZONE STABILITY:")
print(f"   {overlap_top}/10 top zones remain in the top 10 across both years")

# Biggest growers (significant zones)
if len(significant) > 0:
    top_grower = significant.nlargest(1, "growth_pct").iloc[0]
    print(f"\n3. FASTEST GROWING ZONE:")
    print(f"   {top_grower['zone']} ({top_grower['borough']})")
    print(f"   {top_grower['growth_pct']:.1f}% growth")

print(f"\n4. TEMPORAL PATTERNS:")
print(f"   Peak hour 2014: {overlap_2014['hour'].mode().iloc[0]}:00")
print(f"   Peak hour 2015: {overlap_2015['hour'].mode().iloc[0]}:00")
print(f"   Weekday share 2014: {overlap_2014['is_weekday'].mean()*100:.1f}%")
print(f"   Weekday share 2015: {overlap_2015['is_weekday'].mean()*100:.1f}%")

  KEY FINDINGS

1. VOLUME GROWTH (Apr-Jun):
   2014: 1,847,276 pickups
   2015: 7,789,614 pickups
   Growth: 321.7%

2. TOP ZONE STABILITY:
   8/10 top zones remain in the top 10 across both years

3. FASTEST GROWING ZONE:
   Jackson Heights (Queens)
   1432.5% growth

4. TEMPORAL PATTERNS:
   Peak hour 2014: 17:00
   Peak hour 2015: 19:00
   Weekday share 2014: 76.3%
   Weekday share 2015: 69.5%


### Driver Recommendations

Based on the analysis:

1. **Morning rush (7-9 AM):** Position near business districts and transit hubs 
   (Midtown, Financial District, Penn Station area)
2. **Evening rush (5-7 PM):** High demand in office areas — Midtown East/West, 
   Chelsea, Union Square
3. **Late night (10 PM-2 AM):** Focus on nightlife and entertainment districts 
   (Union Sq, TriBeCa, Williamsburg) and residential areas like Park Slope
4. **Weekends:** Demand shifts toward entertainment and residential areas; 
   airports maintain steady volume
5. **Growth zones:** Pay attention to expanding outer-borough areas where competition 
   may be lower but demand is growing

### Limitations

- **Temporal scope:** 2014 covers April-September, 2015 covers January-June. 
  Direct comparison limited to April-June overlap.
- **Data format difference:** 2014 uses GPS (clustered), 2015 uses zone IDs. 
  Comparison at zone level introduces spatial aggregation effects.
- **No external context:** Weather, events, pricing, and competitor data not included.
- **Static analysis:** Hot-zones are averaged over months — real-time demand fluctuates.

### Future Work

- Real-time demand prediction using time-series models
- Weather and event correlation analysis
- Dynamic hot-zone detection (sliding time windows)
- Full-year comparison with seasonality analysis

In [30]:
# Export zone comparison for dashboard
zone_comparison.to_csv(OUTPUT_DIR / "zone_comparison.csv", index=False)
print(f"Saved zone comparison: {len(zone_comparison)} zones -> {OUTPUT_DIR / 'zone_comparison.csv'}")

# Legacy: GPS-level sample (replaced by zone-level aggregation above for choropleth)
# Export for dashboard: 2014 clustered results in CSV (smaller, for Streamlit)
export_cols_2014 = ["datetime", "Lat", "Lon", "hour", "day_of_week", "month",
                     "is_weekday", "cluster", "LocationID", "zone", "borough", "year"]
# Sample for dashboard (full data too large for CSV)
df_2014_dash = df_2014[export_cols_2014].sample(n=min(200_000, len(df_2014)), random_state=42)
df_2014_dash.to_csv(OUTPUT_DIR / "dashboard_2014_sample.csv", index=False)
print(f"Saved 2014 dashboard sample: {len(df_2014_dash):,} rows")

# 2015: aggregate by zone/hour/day for dashboard (not raw data — too large)
agg_2015 = (
    df_2015.groupby(["locationID", "hour", "day_of_week", "month"])
    .size()
    .reset_index(name="count")
)
# Add zone info
col_2015_id = "locationID" if "locationID" in df_2015.columns else "LocationID"
agg_2015 = agg_2015.merge(
    zone_centroids.rename(columns={"LocationID": "locationID"}),
    on="locationID", how="left",
)
agg_2015.to_csv(OUTPUT_DIR / "dashboard_2015_aggregated.csv", index=False)
print(f"Saved 2015 aggregated: {len(agg_2015):,} rows")

# 2014: aggregate by zone for dashboard choropleth (full data, not sampled)
agg_2014 = (
    df_2014.groupby(["LocationID", "hour", "day_of_week", "month"])
    .size()
    .reset_index(name="count")
)
# Dominant cluster per zone (for reference in dashboard)
zone_cluster = (
    df_2014.groupby("LocationID")["cluster"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index(name="dominant_cluster")
)
agg_2014 = agg_2014.merge(zone_cluster, on="LocationID", how="left")
agg_2014 = agg_2014.merge(zone_centroids, on="LocationID", how="left")
agg_2014.to_csv(OUTPUT_DIR / "dashboard_2014_aggregated.csv", index=False)
print(f"Saved 2014 aggregated: {len(agg_2014):,} rows")

print("\nAll data exported for dashboard.")

Saved zone comparison: 259 zones -> /home/sambot/dsfs/000_PROJECTS/UBER/data/output/zone_comparison.csv
Saved 2014 dashboard sample: 200,000 rows
Saved 2015 aggregated: 221,456 rows
Saved 2014 aggregated: 161,597 rows

All data exported for dashboard.


## Next Steps

The Streamlit dashboard provides interactive exploration of these findings with 
temporal filters and year selection.